# Generative Adversarial Network (GAN) Applied to MNIST Dataset

Source:<P>

https://github.com/gordicaleksa/pytorch-GANs/

Adapted:<P>

Antonio Esteves @ UMinho, May 2024<P>

---

In this notebook we will cover the following topics:

✅ What are GANs exactly? <br/>
✅ How to train GANs? <br/>
✅ How to use GANs? <br/>

---

## What is a GAN

GANs were originally proposed by Ian Goodfellow et al. in a paper called [Generative Adversarial Nets](https://papers.nips.cc/paper/5423-generative-adversarial-nets.pdf).


GANs are a framework where 2 models, usually neural networks, called generator (G) and discriminator (D), play a minimax game against each other. The generator is trying to learn the distribution of real data and is the network which we are usually interested in. During the game the goal of the generator is to fool the discriminator into thinking that the images it generates are real. The goal of the discriminator, on the other hand, is to correctly discriminate between the generated (fake) images and real images coming from some dataset (for example, MNIST). 

At the equilibrium of the game the generator learns to generate images indistinguishable from the real images and the best that discriminator can do is output 0.5, meaning it is 50% sure that what you gave him is a real image and 50% sure that it is fake. In this case, it does not have a clue about the images origin. Let us clarify a few concepts.

* **minimax game** is a setup where two players have some goal, or objective function, and one is trying to minimize that objective and the other tries to maximize it.
* **distribution of real data** represents the frequency that each data value occur in our data. We can think of any data sample we use as a point in a **n-dimensional** space. For example, an MNIST mage has size 28x28 and when flattened has 784 numbers. So, an image is simply a point in the 784-dimensional space. As 784-dimensionalsa+pce is hard to imagine, but we can think of it as if it is 3-dimensional or 2-dimensional. So we can think of our data as a 3D or 2D cloud of points. Each point has some probability associated with it, which quantifies how likely is it to appear. If our model has an internal representation of the 3D/2D point cloud, it can generate more points from that cloud. Those new points may correspond to images that did not exist before. Next figure presents an example of a 2-dimensional data distribution.

<img src="../fig/gan_data_distribution.png" alt="Example of a 2D data distribution" align="center" style="width: 550px;"/> <br/>

The height of the plot is the probability of certain datapoint appearing in our data. You can see that points around (0, 0) have the highest probability of happening. Those datapoints could be your 784-dimensional images projected into 2-dimensional space via PCA, t-SNE, or UMAP. In reality, the distribution plot would have multiple peaks, often called a **multi-modal distribution**.

## Import the necessary libraries

In [ ]:
import os
import re
import time
import enum
import cv2               as     cv
import numpy             as     np
import matplotlib.pyplot as     plt
from   git               import Repo

import torch
from   torch                   import nn
from   torch.optim             import Adam
from   torchvision             import transforms, datasets
from   torchvision.utils       import make_grid, save_image
from   torch.utils.data        import DataLoader

from torchmetrics.image.fid       import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

import yaml
import wandb

## Login into Weights & Biases

In [ ]:
wandb.login()

## Read the training session configuration

In [ ]:
COMPUTE_IS  = False
COMPUTE_FID = True

CONFIG_FILE = '../config/gan_v2_04.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [ ]:
print('parameters:')
for key, value in config.items():
    print(f'\t{key}: {value}')

## Track metadata and hyperparameters with Weights & Biases

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb
)

## Define some constants to make the code easier to read

In [ ]:
# Location where the trained models will be saved
MODELS_PATH      = os.path.join(os.getcwd(), 'models')
os.makedirs(MODELS_PATH, exist_ok=True)

# Location where model checkpoints will be saved during training
CHECKPOINTS_PATH   = os.path.join(os.getcwd(), 'models', 'checkpoints')
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

# Location where we will save here the images generated during GAN training
RESULTS_PATH = os.path.join(os.getcwd(), 'results', config["experiment_name"])
os.makedirs(RESULTS_PATH, exist_ok=True)

In [ ]:
print(torch.__version__)

# Setup device agnostic code
device = "cuda:1" if torch.cuda.is_available() else "cpu"

print(f'Using {device} for computing')

repo = Repo.init('OUR_GIT_REPO_PATH')
githash = repo.head.object.hexsha

print(f'GIT HASH: {githash}')

## Exploring the dataset

We should always invest some time to understand our data, in order to answer questions like:
1. How many images do we have?
2. What is the shape of the images?
3. How do the images look like?

So let us answer those questions.

In [ ]:
# Images are usually in the [0.:1.] or [0:255] range. The normalization transform
# will bring them into the [-1, 1] range.

transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((.5,), (.5,))
    ]
)

# MNIST is a simple dataset, so it is included in PyTorch.
# First time we run this notebook, it will download the MNIST dataset and
# store it in 'config["data_dir_root"]' and the 'transform' defined above
# will be applied to every single image of the dataset.

mnist_dataset = datasets.MNIST(
    root      = config["data_dir_root"],
    train     = True,
    download  = True,
    transform = transform,
)

# Create a DataLoader, which is a wrapper class that helps us load images in batches

mnist_data_loader = DataLoader(
    mnist_dataset,
    batch_size = config["batch_size"],
    shuffle    = True,
    drop_last  = True,
)

# How many images has our MNIST dataset?
print(f'Dataset size: {len(mnist_dataset)} images.')

num_imgs_to_visualize = 25  # number of images we will display
batch                 = next(iter(mnist_data_loader))  # draw a single batch from the dataset
img_batch             = batch[0]  # extract only the images and ignore the labels (batch[1])
img_batch_subset      = img_batch[:num_imgs_to_visualize] # extract a subset of images

# What is the shape of the images?
# The format is (BS,C,H,W):
#   BS - number of images in a batch
#   C - number of channels
#   H - height
#   W - width
print(f'Image shape {img_batch_subset.shape[1:]}')  # ignore shape[0], number of images in batch.
# print(img_batch_subset[0])

# How do our images look like?
# Creates a 5x5 grid of images:
#  - we apply normalization to bring images from [-1, 1] range
#    back into [0, 1] for display
#  - pad_value is 1, to fill the grid space between images with white=1 pixels,
#    to de different from the MNIST images background (black=0).
grid = make_grid(
    img_batch_subset,
    nrow      = int(np.sqrt(num_imgs_to_visualize)),
    normalize = True,
    pad_value = 1.,
)
# covert from CxHxW -> HxWxC, which is the format that matplotlib expects
grid = np.moveaxis(grid.numpy(), 0, 2)
plt.figure(figsize=(6, 6))
plt.title("Samples from the MNIST dataset")
plt.imshow(grid)
plt.show()

## Define the GAN model

Let us define the generator and discriminator networks.

The original paper used the maxout activation and dropout for regularization. <P>
We will use `LeakyReLU` instead and `batch normalization` which came after the original paper was published. <P>

Those design decisions are inspired by the DCGAN model which came later than the original GAN.

In [ ]:
# This function creates a batch of random number vectors
def get_gaussian_latent_batch(batch_size, device):
    return torch.randn((batch_size, config["latent_dim"]), device=device)


# This functions creates a GAN building block, which helps to make
# the code cleaner and more modular,
def gan_block(in_features, out_features, normalize=True, activation=None):
    layers = [nn.Linear(in_features, out_features)]
    if normalize:
        layers.append(nn.BatchNorm1d(out_features))
    # DCGAN used LeakyReLU negative slop equal to 0.2
    # Experiments show that values like 0.5 did not make a significant difference
    layers.append(nn.LeakyReLU(negative_slope=0.2) if activation is None else activation)
    return layers

### Generator network

In [ ]:
class Generator(torch.nn.Module):
    """
    The generator is a simple 4-layer MLP neural network.

    By default it works well with MNIST size images (28x28).

    There are many ways you can construct generator to work on MNIST.
    Even without normalization layers it will work ok. Even with 5 layers it will work ok.

    It is generally an open-research question on how to evaluate GANs i.e. quantify the "ok" statement.

    People tried to automate the task using Inception Score (IS) or Frechet Inception Distance (FID),
    but so far it always ends up with some form of visual inspection (human in the loop).
    """
    def __init__(self, img_shape=(config["img_size"], config["img_size"])):
        super().__init__()
        self.generated_img_shape = img_shape

        # These blocks are just linear layers followed by LeakyReLU and batch normalization.
        # Except for the last layer where we exclude batch normalization and
        # we add Tanh, to map images into our [-1, 1] range.
        self.net = nn.Sequential(
            *gan_block(config["latent_dim"], config["g_neurons_per_layer"][0]),
            *gan_block(config["g_neurons_per_layer"][0], config["g_neurons_per_layer"][1]),
            *gan_block(config["g_neurons_per_layer"][1], config["g_neurons_per_layer"][2]),
            *gan_block(config["g_neurons_per_layer"][2], img_shape[0] * img_shape[1],
                normalize=False, activation=nn.Tanh())
        )

    def forward(self, latent_vector_batch):
        img_batch_flattened = self.net(latent_vector_batch)
        # Unflatten using the 'view' method into the (BS, 1, 28, 28) shape of MNIST
        return img_batch_flattened.view(
            img_batch_flattened.shape[0],
            1,
            *self.generated_img_shape
        )

### Discriminator network

In [ ]:
# We can interpret the output from the discriminator as the probability 
# of input image to be from MNIST dataset.
# If the outputs is 1, the discriminator is 100% sure that it is real.
class Discriminator(torch.nn.Module):
    """
    The discriminator is a simple 3-layer MLP neural network.
    It should output probability 1 for real images and 0 for fakes.

    By default it works with MNIST size images (28x28).

     Using normalization as in the DCGAN paper doesnot work well here.
    """
    def __init__(self, img_shape=(config["img_size"], config["img_size"])):
        super().__init__()

        # Last layer is a sigmoid function to discriminate the type of images (0-fake,1-real).
        self.net = nn.Sequential(
            *gan_block(img_shape[0] * img_shape[1], config["d_neurons_per_layer"][0], normalize=False),
            *gan_block(config["d_neurons_per_layer"][0], config["d_neurons_per_layer"][1], normalize=False),
            *gan_block(config["d_neurons_per_layer"][1], config["d_neurons_per_layer"][2], normalize=False,
                activation=nn.Sigmoid())
        )

    def forward(self, img_batch):
        # flatten input from (N,1,H,W) into (N, HxW)
        img_batch_flattened = img_batch.view(img_batch.shape[0], -1)
        return self.net(img_batch_flattened)

## Training the GAN

So far we got familiar with data and our models.<br/>
How to actually train our GAN?

We will be using **BCE** (**binary cross-entropy loss**). If we input real images into the discriminator we expect it to output 1. The further away it is from 1 and the closer it is to 0, the more we should penalize it, as it is making a wrong prediction. This is how the BCE loss should be interpreted.<P>

<img src="../fig/gan_cross_entropy_loss.png" alt="BCE loss when the true label is 1." align="left"/>

BCE loss becomes `-log(x)` when the true label is 1. <br/>

Similarly for fake images, the true label is 0, as we want the discriminator to output 0 for fake images, and we want to penalize the generator when it outputs values close to 1. So, we basically want to mirror the above loss function and that is just: `-log(1-x)`. <br/>

So, BCE loss becomes `-log(1-x)` when the true label is 0.<br/>

### Training utility functions
Let us define some useful utility functions.

In [ ]:
# Training with SGD causes problems to the discriminator optimization.
# Adam works nicely but the default learning rate 1e-3 do not work.
# We have to train the discriminator more than generator, a 4 to 1 scheme
# works with the default learning rat, but still produces worse results.
# LR=0.0002, beta1=0.5, and beta2=0.999 are from the DCGAN paper and 
# work fine here.

def get_optimizers(discriminator, generator):
    d_optimizer = Adam(
        discriminator.parameters(),
        lr    = config["lr"], 
        betas = (config["beta1"], config["beta2"])
    )
    g_optimizer = Adam(
        generator.parameters(), 
        lr    = config["lr"], 
        betas = (config["beta1"], config["beta2"])
    )
    return d_optimizer, g_optimizer

# It is useful to add some metadata when saving the model.
# It probably maked sense to also add the number of epochs.

def get_training_state(generator, discriminator, gan_type_name):
    training_state = {
        "commit_hash":   githash,
        "generator":     generator.state_dict(),
        "discriminator": discriminator.state_dict(),
        "gan_type":      gan_type_name,
    }
    return training_state

# Makes things useful when you have multiple models.

class GANType(enum.Enum):
    GAN   = 0,
    DCGAN = 1,

In [ ]:
def time_format(seconds: int) -> str:
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'

### Track the model progress during training

We can track how the GAN training is progressing through:
1. Console output
2. Images dumped to `data/data_debug`
3. Tensorboard, just type `tensorboard --logdir=runs` in the terminal. 

Note: to use tensorboard just navigate to the project root and open `http://localhost:6006/` in the browser.

In [ ]:
# Generate a batch of reference noise vectors that is kept fixed throughout
# training, to make it easier to assess how the generator capacity evolves.

ref_batch_size  = 16
ref_noise_batch = get_gaussian_latent_batch(
    ref_batch_size,
    device
)

# Lists to store the G and D  losses
discriminator_loss_values = []
generator_loss_values     = []
is_mean_values            = []
is_stddev_values          = []
fid_values                = []

# Constants
img_cnt = 0

# Evaluation metrics: IS and FID
if COMPUTE_IS == True:
    inception = InceptionScore(compute_on_cpu=True).to(device)
if COMPUTE_FID == True:
    fid       = FrechetInceptionDistance(feature=192, compute_on_cpu=True).to(device) #  can be 64, 192, 768, or 2048

# Instantiate the generator and discriminator networks and place them on 'device'

discriminator = Discriminator().train().to(device)
generator     = Generator().train().to(device)

# Select the generator and discriminator optimizers

discriminator_opt, generator_opt = get_optimizers(discriminator, generator)

# When we use 'real_images_gt' as true labels, the BCELoss will be -log(x).
# whereas, when we use 'fake_images_gt' as true labels, the BCELoss will be -log(1-x).

adversarial_loss = nn.BCELoss()
real_images_gt   = torch.ones((config["batch_size"], 1),  device=device)
fake_images_gt   = torch.zeros((config["batch_size"], 1), device=device)

# .......................................................................
# GAN training loop
#
# It is recommended to train the discriminator first, to avoid mode collapse!
# A mode collapse occurs when the generator learns to only generate a single digit
# instead of all 10 digits!

for epoch in range(config["epochs"]):

    ts = time.time()  # start measuring time
    for batch_idx, (real_images, _) in enumerate(mnist_data_loader):

        real_images = real_images.to(device)  # Place images on 'device'

        # Train the discriminator .....................................

        # Maximize L = log(D(x)) + log(1-D(G(z))), or equivalently minimize -L.
        # D = discriminator, x = real images, G = generator, z = latent gaussian vectors, G(z) = fake images

        # Reset the gradients at the batch beginning, since they are accumulated during batch training.
        discriminator_opt.zero_grad()

        # -log(D(x)) <- we minimize this term by making D(x)=Discriminator(real images) as close to 1 as possible
        real_discriminator_loss = adversarial_loss(discriminator(real_images), real_images_gt)

        # Get a batch of random vectors 'z' and pass it through the generator to get a batch of fake images
        # fake_images = G(z)
        fake_images = generator(get_gaussian_latent_batch(config["batch_size"], device))

        # Pass the batch of generated images through the discriminator.
        # fake_images_prediction = D(G(z))
        # We call the 'detach(generator output)' method because we do not want to calculate 
        # gradients for the generator during backpropagation.
        fake_images_predictions = discriminator(fake_images.detach())

        # -log(1 - D(G(z))) <- we minimize this term by making D(G(z)) as close to 0 as possible
        fake_discriminator_loss = adversarial_loss(fake_images_predictions, fake_images_gt)

        discriminator_loss = real_discriminator_loss + fake_discriminator_loss
        discriminator_loss.backward()  # this will calculate gradient for the discriminator weights
        discriminator_opt.step()  # Update discriminator weights using gradients and the optimizer strategy

        # Train the generator ..............................................

        # Minimize L_1 = log(1-D(G(z)))
        # or equivalently maximize L_2 = log(D(G(z)))
        # or equivalently minimize L_3 = -log(D(G(z)))
        # The original L_1 expression has problems with diminishing gradients
        # to apply to the generator when the discriminator is too good.

        # To cause mode collapse an easy way is to optimize the generator several iterations
        # while the discriminator is optimized only once

        # Reset the gradients at the batch beginning, since they are accumulated during batch training.
        generator_opt.zero_grad()

        # Get a batch of random vectors 'z', pass it through the generator to get a batch of fake images,
        # and pass the fake images trough the discriminator.
        # --> D(G(z))
        generated_images_predictions = discriminator(
            generator(get_gaussian_latent_batch(config["batch_size"], device))
        )

        # By using real_images true labels we want to minimize -log(D(G(z))),
        # which occurs when the discriminator outputs '1' (considers fake images real ones),
        # in this way we are fooling the discriminator to thinking the generated images are real.
        generator_loss = adversarial_loss(generated_images_predictions, real_images_gt)

        generator_loss.backward() # this will calculate gradient for the generator weights
        generator_opt.step()  # Update generator weights using gradients and the optimizer strategy

        # Compute IS and FID
        fake_images = fake_images*127.5+127.5
        fake_images = fake_images.to(dtype=torch.uint8, device=device)
        fake_images = fake_images.expand(fake_images.shape[0],3,*fake_images.shape[2:])
        real_images = real_images*127.5+127.5
        real_images = real_images.to(dtype=torch.uint8, device=device)
        real_images = real_images.expand(real_images.shape[0],3,*real_images.shape[2:])

        if COMPUTE_IS == True:
            inception.update(fake_images)
            is_mean, is_stddev = inception.compute()
            gan_is_mean        = is_mean.item()
            gan_is_stddev      = is_stddev.item()
        
        if COMPUTE_FID == True:
            fid.update(real_images, real=True)
            fid.update(fake_images, real=False)
            gan_fid = fid.compute()

        # Save metrics for training monitoring and plotting
        generator_loss_values.append(generator_loss.item())
        discriminator_loss_values.append(discriminator_loss.item())
        if COMPUTE_IS == True:
            is_mean_values.append(gan_is_mean)
            is_stddev_values.append(gan_is_stddev)
        if COMPUTE_FID == True:
            fid_values.append(gan_fid.item())

        # Print progress info, save generated images, save checkpoint creation

        if batch_idx % config["log_interval"] == 0:

            mean_d_loss    = np.mean(discriminator_loss_values[-config["log_interval"]:])
            mean_g_loss    = np.mean(generator_loss_values[-config["log_interval"]:])
            if COMPUTE_IS == True:
                mean_is_mean   = np.mean(is_mean_values[-config["log_interval"]:])
                mean_is_stddev = np.mean(is_stddev_values[-config["log_interval"]:])
            if COMPUTE_FID == True:
                mean_fid       = np.mean(fid_values[-config["log_interval"]:])

            print(f'Epoch training time: {(time.time() - ts):.2f} s | epoch: {epoch + 1} | ', end=" ")
            print(f'batch: {batch_idx + 1}/{len(mnist_data_loader)}', end="  ")
            print(f'Disc loss: {mean_d_loss :0>10.7f}', end="  ")
            print(f'Gen loss:  {mean_g_loss :0>10.7f}', end="  ")
            if COMPUTE_IS == True:
                if COMPUTE_FID == False:
                    print(f'IS: (mean={mean_is_mean :0>7.4f},std={mean_is_stddev :0>7.4f})')
                else:
                    print(f'IS: (mean={mean_is_mean :0>7.4f},std={mean_is_stddev :0>7.4f})', end="  ")
            if COMPUTE_FID == True:
                print(f'FID: {mean_fid :0>10.7f}')

            try:
                # Log metrics to W&B
                if COMPUTE_IS == True and COMPUTE_FID == True:
                    wandb.log(
                        {
                        "discriminator_loss": mean_d_loss,
                        "generator_loss":     mean_g_loss,
                        "is_mean":            mean_is_mean,
                        "is_stddev":          mean_is_stddev,
                        "fid":                mean_fid,
                        "epoch":              epoch+1,
                        }
                    )
                elif COMPUTE_IS == True and COMPUTE_FID == False:
                    wandb.log(
                        {
                        "discriminator_loss": mean_d_loss,
                        "generator_loss":     mean_g_loss,
                        "is_mean":            mean_is_mean,
                        "is_stddev":          mean_is_stddev,
                        "epoch":              epoch+1,
                        }
                    )
                elif COMPUTE_IS == False and COMPUTE_FID == True:
                    wandb.log(
                        {
                        "discriminator_loss": mean_d_loss,
                        "generator_loss":     mean_g_loss,
                        "fid":                mean_fid,
                        "epoch":              epoch+1,
                        }
                    )
                else:
                    wandb.log(
                        {
                        "discriminator_loss": mean_d_loss,
                        "generator_loss":     mean_g_loss,
                        "epoch":              epoch+1,
                        }
                    )
            except Exception as ex:
                print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

        # Save intermediate generated images
        if batch_idx % config["sampling_interval"] == 0:
            with torch.no_grad():
                log_generated_images = generator(ref_noise_batch)
                log_generated_images_resized = nn.Upsample(scale_factor=2.5, mode='nearest')(log_generated_images)
                out_path = os.path.join(RESULTS_PATH, f'{config["experiment_name"]}_generated_from_refZ_{str(img_cnt).zfill(6)}.jpg')
                save_image(log_generated_images_resized, out_path, nrow=int(np.sqrt(ref_batch_size)), normalize=True)
                img_cnt += 1

        # Save a generator and discriminator checkpoint
        if (epoch + 1) % config["checkp_interval"] == 0 and batch_idx == 0:
            ckpt_model_name = f"gan_checkpoint_epoch_{epoch + 1}_batch_{batch_idx + 1}.pth"
            torch.save(
                get_training_state(generator, discriminator, GANType.GAN.name),
                os.path.join(CHECKPOINTS_PATH, ckpt_model_name)
            )

    te  = time.time()
    texec_sec = te - ts
    texec_str = time_format(texec_sec)

    print(f'Epoch training time: {texec_str}')

    try:
        wandb.log(
            {
            "epoch_training_time_sec": texec_sec,
            }
        )
    except Exception as ex:
        print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

# Save the final generator+discriminator in the 'models' directory
torch.save(
    get_training_state(generator, discriminator, GANType.GAN.name),
    os.path.join(MODELS_PATH, f'{config["experiment_name"]}.pth')
)

In [ ]:
if COMPUTE_IS == True and COMPUTE_FID == True:
    nrows = 4
elif COMPUTE_IS == True and COMPUTE_FID == False:
    nrows = 3
elif COMPUTE_IS == False and COMPUTE_FID == True:
    nrows = 3
else:
    nrows = 2

fig, ax = plt.subplots(ncols=1, nrows=nrows, figsize=(6, 6), layout="constrained")
ax[0].plot(discriminator_loss_values)
ax[0].set_title('Discriminator loss')
ax[1].plot(generator_loss_values)
ax[1].set_title('Generator loss')

if COMPUTE_IS == True and COMPUTE_FID == True:
    ax[2].plot(is_mean_values)
    ax[2].set_title('Inception score mean')
    ax[3].plot(fid_values)
    ax[3].set_title('FID')
elif COMPUTE_IS == True and COMPUTE_FID == False:
    ax[2].plot(is_mean_values)
    ax[2].set_title('Inception score mean')
elif COMPUTE_IS == False and COMPUTE_FID == True:
    ax[2].plot(fid_values)
    ax[2].set_title('FID')

## Generate images with our GAN

Finally we can use the trained generator to generate some MNIST-like images.

Let us define a couple of utility functions which will make things cleaner.

In [ ]:
def postprocess_generated_img(generated_img_tensor):
    '''
    Process a generated image, which must be in a Tensor.
    This includes move the tensor to CPU, convert the tensor to numpy array,
    reshape the array from CxHxW to HxWxC, and convert the values from
    [-1, 1] range to [0, 1].
    '''
    assert isinstance(generated_img_tensor, torch.Tensor), \
        f'Expected PyTorch tensor but got {type(generated_img_tensor)}.'

    # Move the tensor from 'device' to CPU, convert it to a numpy array, extract 0th batch, 
    # move the image channel from 0th to 2nd position (CHW -> HWC)
    generated_img = np.moveaxis(generated_img_tensor.to('cpu').numpy()[0], 0, 2)

    # Since MNIST images are grayscale (1-channel only) repeat 3 times to get RGB image
    generated_img = np.repeat(generated_img,  3, axis=2)

    # Convert images from the [-1, 1] range, since the generator has tanh at its output,
    # into the [0, 1] range
    generated_img -= np.min(generated_img)
    generated_img /= np.max(generated_img)

    return generated_img


def generate_from_random_latent_vector(generator):
    '''
    This function will generate a random vector, pass it to the generator
    that will generate a new image, post-process the generated image and return it.
    '''
    with torch.no_grad():  # Do not compute gradients

        # Create a single random (latent) vector and generate an image with generator
        latent_vector = get_gaussian_latent_batch(1, next(generator.parameters()).device)

        # Post process the generator output to covert [-1, 1] range to [0, 1] range
        generated_img = postprocess_generated_img(generator(latent_vector))

    return generated_img


def save_and_maybe_display_image(file_name, img, out_res=(256, 256), should_display=False):
    '''
    Save to file a given image that is stored in a numpy array.
    Optionally, display the image.
    '''
    assert isinstance(img, np.ndarray), f'Expected numpy array but got {type(img)}.'

    # step 1: convert to uint8 format, since OpenCV expects it, otherwise the image will be completely black
    if img.dtype != np.uint8:
        img = (img*255).astype(np.uint8)

    # step 2: write image to the file system; we use [::-1] because opencv expects BGR, not RGB format
    cv.imwrite(
        file_name,
        cv.resize(img[:, :, ::-1], out_res, interpolation=cv.INTER_NEAREST)
    )

    # step 4: display part of the function if desired
    if should_display:
        plt.imshow(img)
        plt.show()

def generate_plot_save_grid_images(generator, save_file_name, grid_H_W=8):
    '''
    Generates a grid of images with the generator, plots the images,
    and saves the grid to a single file.
    '''
    for f in range(grid_H_W**2):
        c = f % grid_H_W
        r = f // grid_H_W

        if c == 0:
            row = generate_from_random_latent_vector(generator)
        else:
            img = generate_from_random_latent_vector(generator)
            row = np.concatenate((row, img), axis=1)
            if c==(grid_H_W-1) and r==0:
                all_imgs = row
            elif c==(grid_H_W-1) and r != 0:
                all_imgs = np.concatenate((all_imgs, row), axis=0)

    plt.figure(figsize=(10, 10), constrained_layout=True)
    plt.imshow(all_imgs)
    plt.imsave(save_file_name, all_imgs)

### Generate new MNIST-like images

In [ ]:
# Define the model file name
model_path = os.path.join(MODELS_PATH, f'{config["experiment_name"]}.pth')
assert os.path.exists(model_path), f'Could not find the model {model_path}!'

# Select the computing device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Let us load the model, this is a dictionary containing model weights but also some metadata.
# commit_hash - tells us which version of the code generated this model.
# gan_type    - this one is "GAN" but there is also "DCGAN" and "CGAN" models.
# state_dict  - contains the actual neural network weights.
model_state = torch.load(model_path)
print(f'Model state contains this data: {model_state.keys()}')

gan_type = model_state["gan_type"]
print(f'Using {gan_type} GAN!')

# Instantiate the generator and place it on 'device'
generator = Generator().to(device)

# Load the weights; strict=True makes sure that the architecture corresponds 
# to the loaded weights 100%
generator.load_state_dict(model_state["generator"], strict=True)

# Put the model in evaluation mode
generator.eval()

# Reset the counter of generated grid of images
GRID_ID = 0

In [ ]:
# Generate a grid of new images

grid_H_W = 8 # grid size

f_name = f'{RESULTS_PATH}/{config["experiment_name"]}_generated_{str(GRID_ID).zfill(3)}.jpg'

print(f'Generating a grid of {grid_H_W} x {grid_H_W} new MNIST-like images')

generate_plot_save_grid_images(generator, f_name, grid_H_W)

GRID_ID += 1

In [ ]:
# Mark the W&B run as finished
wandb.finish()